# IEEE-CIS Fraud Detection — Team 2 / Member 4 (TabNet)

Colab-first **TabNet** baseline on merged Team 1 parquet:

- Chronological **train/valid** split on `TransactionDT` (same idea as Member 3)
- **Median** imputation for numerics and **frequency-bucket** categorical codes (fit on train split only)
- `pytorch-tabnet` **TabNetClassifier** with GPU when available
- Validation **ROC-AUC**, **feature importance**, optional **explain** sample, test predictions + metrics on Drive

**Single-session Colab:** preprocessing is **bundled** in this notebook (same source as `src/modeling/tabnet_preprocess.py` in the repo). It is written to `/content/tabnet_preprocess.py` at runtime so you do **not** need to upload the repo for that file.

## 1. Runtime

1. Colab: **GPU** runtime.
2. Merged parquet: `ieee_cis_fraud/merged/train_member1_member2_merged.parquet` (and test).
3. Run top to bottom.

In [1]:
from pathlib import Path

RANDOM_SEED = 42
TARGET_COLUMN = "isFraud"
ID_COLUMN = "TransactionID"
VALID_SIZE = 0.2
SMOKE_TEST = False
SMOKE_ROWS = 50_000
MAX_CAT = 64

DATA_ROOT = Path("/content")
# MERGED_DIR = DATA_ROOT / "merged"
OUTPUT_DIR = DATA_ROOT / "member4_tabnet"
TRAIN_PATH = DATA_ROOT / "train_member1_member2_merged.parquet"
TEST_PATH = DATA_ROOT / "test_member1_member2_merged.parquet"

# Example: To save the model to Google Drive, uncomment and modify the line below
# Make sure your Google Drive is mounted (as done in cell kem-vnE2Uckw)
# MODEL_DIR = Path("/content/drive/MyDrive/your_project_folder/member4_tabnet_model")
MODEL_DIR = OUTPUT_DIR / "member4_tabnet_model"
METRICS_PATH = OUTPUT_DIR / "member4_tabnet_metrics.json"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "member4_tabnet_feature_importance.csv"
EXPLAIN_SAMPLE_PATH = OUTPUT_DIR / "member4_tabnet_explain_valid_sample.csv"
VALID_PREDICTIONS_PATH = OUTPUT_DIR / "member4_tabnet_valid_predictions.parquet"
TEST_PREDICTIONS_PATH = OUTPUT_DIR / "member4_tabnet_test_predictions.parquet"

TRAIN_PATH, TEST_PATH, OUTPUT_DIR

(PosixPath('/content/train_member1_member2_merged.parquet'),
 PosixPath('/content/test_member1_member2_merged.parquet'),
 PosixPath('/content/member4_tabnet'))

In [2]:
!pip -q install pytorch-tabnet torch pyarrow pandas scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 3.7 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Mounted at /content/drive
TRAIN_PATH: /content/train_member1_member2_merged.parquet
TEST_PATH: /content/test_member1_member2_merged.parquet
OUTPUT_DIR: /content/member4_tabnet


In [4]:
import gc
import importlib.util
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import roc_auc_score

# Bundled copy of src/modeling/tabnet_preprocess.py
_TABNET_PREPROCESS_PATH = Path("/content/tabnet_preprocess.py")
_TABNET_PREPROCESS_SOURCE = r'''
"""
TabNet-oriented preprocessing for merged IEEE-CIS parquet features.

Fits on training split only (median imputation, categorical bucketing).
Outputs float32 matrices with categorical columns as integer codes at fixed tail indices.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer


@dataclass
class TabNetPreprocessor:
    """Transform train/valid/test DataFrames into TabNet-ready numpy arrays."""

    feature_columns: list[str]
    numeric_columns: list[str]
    categorical_columns: list[str]
    imputer: SimpleImputer
    cat_transformers: dict[str, Callable[[pd.Series], np.ndarray]]
    cat_dims: list[int]

    @classmethod
    def fit(cls, X_train: pd.DataFrame, *, max_categories: int = 64) -> TabNetPreprocessor:
        numeric_columns, categorical_columns = _detect_column_types(X_train)
        feature_columns = numeric_columns + categorical_columns

        imputer = SimpleImputer(strategy="median")
        imputer.fit(X_train[numeric_columns].astype("float64"))

        cat_transformers: dict[str, Callable[[pd.Series], np.ndarray]] = {}
        cat_dims: list[int] = []

        for col in categorical_columns:
            mapping, dim, unk_idx = _fit_category_mapping(
                X_train[col], max_categories=max_categories
            )
            cat_transformers[col] = _make_cat_transformer(mapping, unk_idx)
            cat_dims.append(dim)

        return cls(
            feature_columns=feature_columns,
            numeric_columns=numeric_columns,
            categorical_columns=categorical_columns,
            imputer=imputer,
            cat_transformers=cat_transformers,
            cat_dims=cat_dims,
        )

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        X = X.reindex(columns=self.feature_columns)
        num = self.imputer.transform(X[self.numeric_columns].astype("float64")).astype(np.float32)
        if not self.categorical_columns:
            return num.astype(np.float32)
        cat_blocks = [
            self.cat_transformers[c](X[c]).reshape(-1, 1).astype(np.float32)
            for c in self.categorical_columns
        ]
        return np.hstack([num, *cat_blocks], dtype=np.float32)

    @property
    def cat_idxs(self) -> list[int]:
        start = len(self.numeric_columns)
        return list(range(start, start + len(self.categorical_columns)))


def _detect_column_types(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_cols: list[str] = []
    cat_cols: list[str] = []
    for col in df.columns:
        if pd.api.types.is_bool_dtype(df[col]):
            numeric_cols.append(col)
        elif df[col].dtype == object or str(df[col].dtype).startswith("string"):
            cat_cols.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)
        else:
            cat_cols.append(col)
    return numeric_cols, cat_cols


def _fit_category_mapping(series: pd.Series, *, max_categories: int) -> tuple[dict[str, int], int, int]:
    s = series.astype("string").fillna("__MISSING__")
    vc = s.value_counts()
    top_n = max(1, max_categories - 1)
    top_values = vc.head(top_n).index.tolist()
    mapping: dict[str, int] = {}
    for idx, val in enumerate(top_values):
        mapping[str(val)] = idx
    unk_idx = len(top_values)
    dim = unk_idx + 1
    return mapping, dim, unk_idx


def _make_cat_transformer(mapping: dict[str, int], unk_idx: int) -> Callable[[pd.Series], np.ndarray]:
    def _encode(series: pd.Series) -> np.ndarray:
        s = series.astype("string").fillna("__MISSING__").map(lambda x: str(x))
        return s.map(lambda v: mapping.get(v, unk_idx)).astype(np.int64).to_numpy()

    return _encode
'''
_TABNET_PREPROCESS_PATH.write_text(_TABNET_PREPROCESS_SOURCE, encoding="utf-8")

spec = importlib.util.spec_from_file_location("tabnet_preprocess", _TABNET_PREPROCESS_PATH)
mod = importlib.util.module_from_spec(spec)
assert spec.loader is not None

# Fix: Register module in sys.modules BEFORE executing it so @dataclass can find it
sys.modules[spec.name] = mod
spec.loader.exec_module(mod)

TabNetPreprocessor = mod.TabNetPreprocessor

def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024**2

print("Wrote", _TABNET_PREPROCESS_PATH)
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())

Wrote /content/tabnet_preprocess.py
Torch: 2.10.0+cu128 CUDA: True


In [6]:
assert TRAIN_PATH.exists(), TRAIN_PATH
# assert TEST_PATH.exists(), TEST_PATH

train_df = pd.read_parquet(TRAIN_PATH)
# test_df = pd.read_parquet(TEST_PATH)

if SMOKE_TEST:
    train_df = train_df.sample(n=min(SMOKE_ROWS, len(train_df)), random_state=RANDOM_SEED).reset_index(drop=True)
    # test_df = test_df.sample(n=min(SMOKE_ROWS, len(test_df)), random_state=RANDOM_SEED).reset_index(drop=True)

# print(train_df.shape, test_df.shape)
print(train_df.shape)
print("train MB:", round(memory_usage_mb(train_df), 2))

(590540, 151)
train MB: 1095.97


In [7]:
assert TARGET_COLUMN in train_df.columns
# assert TARGET_COLUMN not in test_df.columns
assert "TransactionDT" in train_df.columns

y = train_df[TARGET_COLUMN].astype(np.int64)
train_ids = train_df[ID_COLUMN].copy()
# test_ids = test_df[ID_COLUMN].copy()

X = train_df.drop(columns=[TARGET_COLUMN])
# X_test_raw = test_df.copy()

order = X["TransactionDT"].sort_values(kind="mergesort").index
X = X.loc[order].reset_index(drop=True)
y = y.loc[order].reset_index(drop=True)
train_ids = train_ids.loc[order].reset_index(drop=True)

split_idx = int(len(X) * (1 - VALID_SIZE))
X_train_df = X.iloc[:split_idx].copy()
X_valid_df = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].to_numpy(dtype=np.int64)
y_valid = y.iloc[split_idx:].to_numpy(dtype=np.int64)
valid_ids = train_ids.iloc[split_idx:].reset_index(drop=True)

X_train_fit = X_train_df.drop(columns=[ID_COLUMN], errors="ignore")
X_valid_fit = X_valid_df.drop(columns=[ID_COLUMN], errors="ignore")
# X_test_fit = X_test_raw.drop(columns=[ID_COLUMN], errors="ignore")

del train_df, X, y, train_ids
gc.collect()

print("X_train:", X_train_fit.shape, "X_valid:", X_valid_fit.shape)
print("fraud rate train/valid:", y_train.mean(), y_valid.mean())

X_train: (472432, 149) X_valid: (118108, 149)
fraud rate train/valid: 0.03513521522674162 0.034409184813899145


In [8]:
pre = TabNetPreprocessor.fit(X_train_fit, max_categories=MAX_CAT)

X_train = pre.transform(X_train_fit)
X_valid = pre.transform(X_valid_fit)
# X_test = pre.transform(X_test_fit)

cat_idxs = pre.cat_idxs
cat_dims = pre.cat_dims
feat_names = pre.numeric_columns + pre.categorical_columns

print("n_features:", X_train.shape[1], "cat columns:", len(cat_idxs), "cat_dims sample:", cat_dims[:5])
assert np.isfinite(X_train).all() and np.isfinite(X_valid).all()

n_features: 149 cat columns: 31 cat_dims sample: [6, 6, 61, 62, 4]


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
emb_dim = min(16, max(2, MAX_CAT // 4))

tab_kwargs = dict(
    n_d=32,
    n_a=32,
    n_steps=4,
    gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    scheduler_params={"step_size": 25, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type="sparsemax",
    seed=RANDOM_SEED,
    verbose=1,
    device_name=device,
)
if cat_idxs:
    tab_kwargs["cat_idxs"] = cat_idxs
    tab_kwargs["cat_dims"] = cat_dims
    tab_kwargs["cat_emb_dim"] = emb_dim

clf = TabNetClassifier(**tab_kwargs)

clf.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=["valid"],
    eval_metric=["auc"],
    max_epochs=80,
    patience=12,
    batch_size=4096,
    virtual_batch_size=256,
    num_workers=0,
    drop_last=False,
)

print("Best epoch:", clf.best_epoch)

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.15644 | valid_auc: 0.70857 |  0:00:16s
epoch 1  | loss: 0.12845 | valid_auc: 0.79126 |  0:00:31s
epoch 2  | loss: 0.12203 | valid_auc: 0.80848 |  0:00:46s
epoch 3  | loss: 0.11528 | valid_auc: 0.81503 |  0:01:01s
epoch 4  | loss: 0.1124  | valid_auc: 0.81747 |  0:01:16s
epoch 5  | loss: 0.10701 | valid_auc: 0.83497 |  0:01:32s
epoch 6  | loss: 0.10447 | valid_auc: 0.83481 |  0:01:47s
epoch 7  | loss: 0.1003  | valid_auc: 0.83636 |  0:02:02s
epoch 8  | loss: 0.09855 | valid_auc: 0.843   |  0:02:17s
epoch 9  | loss: 0.09578 | valid_auc: 0.83348 |  0:02:32s
epoch 10 | loss: 0.09457 | valid_auc: 0.84472 |  0:02:50s
epoch 11 | loss: 0.09507 | valid_auc: 0.84812 |  0:03:05s
epoch 12 | loss: 0.09186 | valid_auc: 0.84982 |  0:03:20s
epoch 13 | loss: 0.08966 | valid_auc: 0.85003 |  0:03:36s
epoch 14 | loss: 0.08882 | valid_auc: 0.84926 |  0:03:51s
epoch 15 | loss: 0.0873  | valid_auc: 0.85255 |  0:04:06s
epoch 16 | loss: 0.08507 | valid_auc: 0.84778 |  0:04:21s
epoch 17 | los

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Best epoch: 17


In [11]:
clf.save_model(str(MODEL_DIR))
print(f"Model saved to: {MODEL_DIR}")

Successfully saved model at /content/member4_tabnet/member4_tabnet_model.zip
Model saved to: /content/member4_tabnet/member4_tabnet_model


In [10]:
valid_probs = clf.predict_proba(X_valid)[:, 1]
valid_auc = roc_auc_score(y_valid, valid_probs)
print("Validation ROC-AUC:", round(float(valid_auc), 6))

pd.DataFrame({
    ID_COLUMN: valid_ids.astype(np.int64),
    "prob_fraud": valid_probs.astype(np.float32),
}).to_parquet(VALID_PREDICTIONS_PATH, index=False)

importances = getattr(clf, "feature_importances_", None)
if importances is not None:
    pd.DataFrame({"feature": feat_names, "importance": importances}).sort_values(
        "importance", ascending=False
    ).to_csv(FEATURE_IMPORTANCE_PATH, index=False)

explain_n = min(8192, len(X_valid))
try:
    explain_mtx, _masks = clf.explain(X_valid[:explain_n])
    mean_imp = np.abs(np.asarray(explain_mtx)).mean(axis=0)
    pd.DataFrame({"feature": feat_names, "mean_abs_explain": mean_imp}).sort_values(
        "mean_abs_explain", ascending=False
    ).to_csv(EXPLAIN_SAMPLE_PATH, index=False)
except Exception as exc:
    print("explain() skipped:", exc)

Validation ROC-AUC: 0.862468


In [ ]:
# Load the model
loaded_clf = TabNetClassifier()
loaded_clf.load_model(str(MODEL_DIR))
print(f"Model loaded from: {MODEL_DIR}")

In [ ]:
# test_probs = clf.predict_proba(X_test)[:, 1]
# pd.DataFrame({
#     ID_COLUMN: test_ids.astype(np.int64),
#     "prob_fraud": test_probs.astype(np.float32),
# }).to_parquet(TEST_PREDICTIONS_PATH, index=False)

# clf.save_model(str(MODEL_DIR))

# payload = {
#     "valid_auc": float(valid_auc),
#     "best_epoch": int(clf.best_epoch),
#     "device": device,
#     "n_features": int(X_train.shape[1]),
#     "n_categorical": len(cat_idxs),
#     "max_categories": MAX_CAT,
#     "train_rows": int(X_train.shape[0]),
#     "valid_rows": int(X_valid.shape[0]),
#     "model_dir": str(MODEL_DIR),
# }
# METRICS_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
# print("Wrote:", METRICS_PATH, MODEL_DIR, TEST_PREDICTIONS_PATH)